# The Edge — GraphSAGE Node Classification
## Predicting Room Types Across 8 Residential Units

**Building:** The Edge, Carrasco, Montevideo, Uruguay  
**Architects:** Foster + Partners + Ponce de León Architects  
**Student:** Rania Chihaoui | IAAC 2025–26

---

### What this notebook does
1. Encodes all 8 residential units as room-level graphs from the official brochure floor plans
2. Exports a dataset in **Modified Swiss Dwellings (MSD)** CSV schema
3. Combines The Edge graphs with the MSD training dataset
4. Trains a **GraphSAGE** model for node classification on MSD data
5. Predicts room types for The Edge and analyses the results spatially

---

### Label → Room Type Reference (MSD Schema)

| Label | Zone | Room Type | The Edge rooms |
|-------|------|-----------|----------------|
| 0 | Day zone | Living / Dining | Estar, Comedor, Solárium, Terraza, Jardín |
| 1 | Night zone | Bedroom | Dormitorio, Dormitorio Servicio |
| 2 | Night zone | Master bedroom | Dormitorio Principal (with vestidor) |
| 3 | Night zone | Study / small room | Vestidor — **rare in MSD (0.26% of nodes)** |
| 4 | Night zone | Secondary bedroom | *(not used in this project)* |
| 5 | Service zone | Kitchen / Utility | Cocina, Kitchenette, Lavadero |
| 6 | Service zone | WC / Powder room | Toilette |
| 7 | Service zone | Bathroom | Baño, Jacuzzi |
| 8 | Circulation | Corridor / Hallway | Pasillo |

## Section 1 — Setup

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

renderer = "vscode"

BASE_DIR  = Path(r"c:\Users\Win11\GraphML_RaniaChihaoui")
MSD_PATH  = Path(r"C:\Users\Win11\Desktop\AIA2026\GraphLM\Updated Notebooks\Graph ML -- NOTEBOOKS\Supporting Files\dataset_node_classification")
EDGE_PATH = BASE_DIR / "dataset_the_edge"
COMB_PATH = BASE_DIR / "dataset_combined"

EDGE_PATH.mkdir(parents=True, exist_ok=True)
COMB_PATH.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = {
    0: "Day (Living/Dining/Terrace)",
    1: "Night (Bedroom)",
    2: "Night (Master Bedroom)",
    3: "Night (Walk-in Closet)",
    4: "Night (Service Bedroom)",
    5: "Service (Kitchen/Laundry)",
    6: "Service (WC/Toilette)",
    7: "Service (Bathroom)",
    8: "Circulation (Corridor)",
}

print(f"topologicpy version: {Helper.Version()}")
print(f"MSD dataset:      {MSD_PATH}")
print(f"TheEdge dataset:  {EDGE_PATH}")
print(f"Combined dataset: {COMB_PATH}")

## Section 2 — Room Program (from Brochure)

Each unit is one graph. Each room is one node with a label (0–8) matching the MSD schema.
Adjacencies (edges) are read directly from the floor plan: rooms that share a door are connected.

In [ ]:
UNIT_ROOMS = {
    # ── UNIT 101 — DUPLEX (GF + F1) ─────────────────────────────────────────
    # 479 m² covered · 573 m² private garden
    # GF: family living, 2 master bedrooms + walk-ins, service bedroom, laundry, garden
    # F1: main living, dining, kitchen, powder room, laundry, terrace
    "101": {
        "graph_id": 5001, "unit_type": "DUPLEX", "floors": [0, 1],
        "rooms": [
            # GF — nodes 0-11
            {"name": "Estar Familiar GF",      "floor": 0, "label": 0, "area": 55},
            {"name": "Pasillo GF",             "floor": 0, "label": 8, "area": 15},
            {"name": "Dormitorio Principal 1", "floor": 0, "label": 2, "area": 30},
            {"name": "Baño Principal 1",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 1",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Principal 2", "floor": 0, "label": 2, "area": 28},
            {"name": "Baño Principal 2",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 2",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Servicio",    "floor": 0, "label": 1, "area": 18},
            {"name": "Baño Servicio",          "floor": 0, "label": 7, "area":  6},
            {"name": "Lavadero GF",            "floor": 0, "label": 5, "area": 10},
            {"name": "Jardín Privado",         "floor": 0, "label": 0, "area": 80},
            # F1 — nodes 12-17
            {"name": "Estar F1",               "floor": 1, "label": 0, "area": 45},
            {"name": "Comedor",                "floor": 1, "label": 0, "area": 30},
            {"name": "Cocina F1",              "floor": 1, "label": 5, "area": 20},
            {"name": "Toilette F1",            "floor": 1, "label": 6, "area":  5},
            {"name": "Lavadero F1",            "floor": 1, "label": 5, "area":  8},
            {"name": "Terraza F1",             "floor": 1, "label": 0, "area": 40},
        ],
        "edges": [
            (0,1),(0,11),(1,2),(1,5),(1,8),(1,10),
            (2,3),(2,4),(5,6),(5,7),(8,9),
            (12,13),(12,17),(13,14),(14,15),(14,16),
            (0,12),  # private stair GF ↔ F1
        ],
    },
    # ── UNIT 102 — 1 PLANTA (F1 only) ────────────────────────────────────────
    # 269 m² covered
    "102": {
        "graph_id": 5002, "unit_type": "1 PLANTA", "floors": [1],
        "rooms": [
            {"name": "Estar Familiar",         "floor": 1, "label": 0, "area": 60},
            {"name": "Dormitorio Principal",   "floor": 1, "label": 2, "area": 28},
            {"name": "Baño Principal",         "floor": 1, "label": 7, "area":  8},
            {"name": "Vestidor Principal",     "floor": 1, "label": 3, "area": 10},
            {"name": "Dormitorio 2",           "floor": 1, "label": 1, "area": 22},
            {"name": "Baño 2",                "floor": 1, "label": 7, "area":  6},
            {"name": "Cocina",                "floor": 1, "label": 5, "area": 20},
            {"name": "Lavadero",              "floor": 1, "label": 5, "area":  8},
            {"name": "Baño Guest",            "floor": 1, "label": 7, "area":  5},
            {"name": "Terraza",               "floor": 1, "label": 0, "area": 50},
        ],
        "edges": [(0,1),(0,4),(0,6),(0,9),(1,2),(1,3),(4,5),(6,7),(6,8)],
    },
    # ── UNIT 103 — 1 PLANTA (F1 only) ────────────────────────────────────────
    # 286 m² covered
    "103": {
        "graph_id": 5003, "unit_type": "1 PLANTA", "floors": [1],
        "rooms": [
            {"name": "Estar Familiar",         "floor": 1, "label": 0, "area": 65},
            {"name": "Dormitorio Principal",   "floor": 1, "label": 2, "area": 32},
            {"name": "Baño Principal",         "floor": 1, "label": 7, "area":  8},
            {"name": "Vestidor A",             "floor": 1, "label": 3, "area": 10},
            {"name": "Vestidor B",             "floor": 1, "label": 3, "area":  8},
            {"name": "Dormitorio 2",           "floor": 1, "label": 1, "area": 24},
            {"name": "Baño 2",                "floor": 1, "label": 7, "area":  6},
            {"name": "Cocina",                "floor": 1, "label": 5, "area": 20},
            {"name": "Lavadero",              "floor": 1, "label": 5, "area":  8},
            {"name": "Baño Guest",            "floor": 1, "label": 7, "area":  5},
            {"name": "Terraza",               "floor": 1, "label": 0, "area": 60},
        ],
        "edges": [(0,1),(0,5),(0,7),(0,10),(1,2),(1,3),(1,4),(5,6),(7,8),(7,9)],
    },
    # ── UNIT 104 — DUPLEX (GF + F1) ──────────────────────────────────────────
    # 502 m² covered · 482 m² private garden (mirror plan of 101)
    "104": {
        "graph_id": 5004, "unit_type": "DUPLEX", "floors": [0, 1],
        "rooms": [
            {"name": "Estar Familiar GF",      "floor": 0, "label": 0, "area": 55},
            {"name": "Pasillo GF",             "floor": 0, "label": 8, "area": 15},
            {"name": "Dormitorio Principal 1", "floor": 0, "label": 2, "area": 30},
            {"name": "Baño Principal 1",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 1",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Principal 2", "floor": 0, "label": 2, "area": 28},
            {"name": "Baño Principal 2",       "floor": 0, "label": 7, "area":  8},
            {"name": "Vestidor 2",             "floor": 0, "label": 3, "area": 10},
            {"name": "Dormitorio Servicio",    "floor": 0, "label": 1, "area": 18},
            {"name": "Baño Servicio",          "floor": 0, "label": 7, "area":  6},
            {"name": "Lavadero GF",            "floor": 0, "label": 5, "area": 10},
            {"name": "Jardín Privado",         "floor": 0, "label": 0, "area": 80},
            {"name": "Estar F1",               "floor": 1, "label": 0, "area": 45},
            {"name": "Comedor",                "floor": 1, "label": 0, "area": 30},
            {"name": "Cocina F1",              "floor": 1, "label": 5, "area": 20},
            {"name": "Toilette F1",            "floor": 1, "label": 6, "area":  5},
            {"name": "Lavadero F1",            "floor": 1, "label": 5, "area":  8},
            {"name": "Terraza F1",             "floor": 1, "label": 0, "area": 40},
        ],
        "edges": [
            (0,1),(0,11),(1,2),(1,5),(1,8),(1,10),
            (2,3),(2,4),(5,6),(5,7),(8,9),
            (12,13),(12,17),(13,14),(14,15),(14,16),
            (0,12),
        ],
    },
    # ── UNIT 201 — DUPLEX (F2 + Rooftop) ─────────────────────────────────────
    # 261 m² covered · 125 m² rooftop
    "201": {
        "graph_id": 5005, "unit_type": "DUPLEX", "floors": [2, 3],
        "rooms": [
            # F2 — nodes 0-9
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 60},
            {"name": "Pasillo F2",             "floor": 2, "label": 8, "area": 12},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 28},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area":  8},
            {"name": "Vestidor",               "floor": 2, "label": 3, "area": 10},
            {"name": "Dormitorio 2",           "floor": 2, "label": 1, "area": 22},
            {"name": "Baño 2",                "floor": 2, "label": 7, "area":  6},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 18},
            {"name": "Toilette F2",            "floor": 2, "label": 6, "area":  5},
            {"name": "Terraza F2",             "floor": 2, "label": 0, "area": 40},
            # Rooftop — nodes 10-14
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 50},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 15},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 10},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 75},
        ],
        "edges": [
            (0,1),(0,7),(0,9),(1,2),(1,5),(2,3),(2,4),(5,6),(7,8),
            (10,11),(10,12),(10,14),(12,13),
            (0,10),  # private stair F2 ↔ Rooftop
        ],
    },
    # ── UNIT 202 — DUPLEX (F2 + Rooftop) ─────────────────────────────────────
    # 304 m² covered · 152 m² rooftop
    "202": {
        "graph_id": 5006, "unit_type": "DUPLEX", "floors": [2, 3],
        "rooms": [
            # F2 — nodes 0-10
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 70},
            {"name": "Pasillo F2",             "floor": 2, "label": 8, "area": 12},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 32},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area":  9},
            {"name": "Vestidor",               "floor": 2, "label": 3, "area": 12},
            {"name": "Dormitorio 2",           "floor": 2, "label": 1, "area": 25},
            {"name": "Baño 2",                "floor": 2, "label": 7, "area":  7},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 22},
            {"name": "Toilette F2",            "floor": 2, "label": 6, "area":  5},
            {"name": "Lavadero F2",            "floor": 2, "label": 5, "area":  9},
            {"name": "Terraza F2",             "floor": 2, "label": 0, "area": 45},
            # Rooftop — nodes 11-15
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 70},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 18},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 12},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 82},
        ],
        "edges": [
            (0,1),(0,7),(0,10),(1,2),(1,5),(2,3),(2,4),(5,6),(7,8),(7,9),
            (11,12),(11,13),(11,15),(13,14),
            (0,11),
        ],
    },
    # ── UNIT 203 — DUPLEX (F2 + Rooftop) ─────────────────────────────────────
    # 307 m² covered · 160 m² rooftop
    "203": {
        "graph_id": 5007, "unit_type": "DUPLEX", "floors": [2, 3],
        "rooms": [
            # F2 — nodes 0-10
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 65},
            {"name": "Pasillo F2",             "floor": 2, "label": 8, "area": 12},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 30},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area":  8},
            {"name": "Vestidor",               "floor": 2, "label": 3, "area": 10},
            {"name": "Dormitorio 2",           "floor": 2, "label": 1, "area": 24},
            {"name": "Baño 2",                "floor": 2, "label": 7, "area":  7},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 20},
            {"name": "Lavadero F2",            "floor": 2, "label": 5, "area":  8},
            {"name": "Baño Guest",             "floor": 2, "label": 7, "area":  5},
            {"name": "Terraza F2",             "floor": 2, "label": 0, "area": 45},
            # Rooftop — nodes 11-15
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 70},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 18},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 12},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 80},
        ],
        "edges": [
            (0,1),(0,7),(0,10),(1,2),(1,5),(2,3),(2,4),(5,6),(7,8),(7,9),
            (11,12),(11,13),(11,15),(13,14),
            (0,11),
        ],
    },
    # ── UNIT 204 — TRIPLEX (F1 + F2 + Rooftop) ───────────────────────────────
    # 429 m² covered · 177 m² rooftop
    # F1: 2 bedrooms + terrace  |  F2: living, master, kitchen, guest bath
    # Rooftop: solárium, jacuzzi, kitchenette, toilette, terraza
    "204": {
        "graph_id": 5008, "unit_type": "TRIPLEX", "floors": [1, 2, 3],
        "rooms": [
            # F1 — nodes 0-4
            {"name": "Dormitorio 1",           "floor": 1, "label": 1, "area": 30},
            {"name": "Baño 1 F1",              "floor": 1, "label": 7, "area":  8},
            {"name": "Dormitorio 2",           "floor": 1, "label": 1, "area": 28},
            {"name": "Baño 2 F1",              "floor": 1, "label": 7, "area":  8},
            {"name": "Terraza F1",             "floor": 1, "label": 0, "area": 40},
            # F2 — nodes 5-9
            {"name": "Estar Familiar F2",      "floor": 2, "label": 0, "area": 80},
            {"name": "Dormitorio Principal",   "floor": 2, "label": 2, "area": 35},
            {"name": "Baño Principal",         "floor": 2, "label": 7, "area": 10},
            {"name": "Cocina F2",              "floor": 2, "label": 5, "area": 22},
            {"name": "Baño Guest F2",          "floor": 2, "label": 7, "area":  6},
            # Rooftop — nodes 10-14
            {"name": "Solárium",               "floor": 3, "label": 0, "area": 90},
            {"name": "Jacuzzi",                "floor": 3, "label": 7, "area": 18},
            {"name": "Kitchenette Roof",       "floor": 3, "label": 5, "area": 12},
            {"name": "Toilette Roof",          "floor": 3, "label": 6, "area":  5},
            {"name": "Terraza Roof",           "floor": 3, "label": 0, "area": 87},
        ],
        "edges": [
            (0,1),(0,4),(2,3),(2,4),
            (0,5),   # stair F1 → F2
            (5,6),(5,8),(5,9),(6,7),
            (5,10),  # stair F2 → Rooftop
            (10,11),(10,12),(10,14),(12,13),
        ],
    },
}

print(f"{'Unit':<6} {'Type':<10} {'Rooms':>6} {'Edges':>6}")
print("-" * 32)
total_rooms = 0
for uid, u in UNIT_ROOMS.items():
    print(f"{uid:<6} {u['unit_type']:<10} {len(u['rooms']):>6} {len(u['edges']):>6}")
    total_rooms += len(u['rooms'])
print("-" * 32)
print(f"{'TOTAL':<6} {'':<10} {total_rooms:>6}")

## Section 3 — Feature Engineering & CSV Export

Node features must exactly match the MSD schema:
- `feat_zoning_type_0/1/2/3` — one-hot encoding of the 4 broad zone categories
- `feat_connectivity_0/1/2` — set to `[0, 1, 0]` (dominant MSD pattern: standard interior room)

All The Edge nodes receive `test_mask=1` — they are the inference set.

In [ ]:
def label_to_zoning(label):
    """One-hot encoding of the 4 MSD zone categories."""
    z = [0, 0, 0, 0]
    if label == 0:              z[0] = 1   # day zone
    elif label in (1,2,3,4):   z[1] = 1   # night zone
    elif label in (5,6,7):     z[2] = 1   # service zone
    elif label == 8:            z[3] = 1   # circulation zone
    return z


def build_unit_dataframes(unit_id, unit_data):
    rooms = unit_data["rooms"]
    edges = unit_data["edges"]
    gid   = unit_data["graph_id"]

    # ── Nodes ────────────────────────────────────────────────────────────────
    node_rows = []
    for nid, room in enumerate(rooms):
        z = label_to_zoning(room["label"])
        node_rows.append({
            "graph_id":           gid,
            "node_id":            nid,
            "label":              room["label"],
            "feat_zoning_type_0": z[0],
            "feat_zoning_type_1": z[1],
            "feat_zoning_type_2": z[2],
            "feat_zoning_type_3": z[3],
            "feat_connectivity_0": 0,
            "feat_connectivity_1": 1,
            "feat_connectivity_2": 0,
            "train_mask": 0,
            "val_mask":   0,
            "test_mask":  1,
            # metadata columns (stripped before PyG export)
            "_unit":      unit_id,
            "_room_name": room["name"],
            "_floor":     room["floor"],
            "_area":      room["area"],
        })
    nodes_df = pd.DataFrame(node_rows)

    # ── Edges (undirected → both directions, matching MSD [0,1,0] encoding) ─
    edge_rows = []
    for (src, dst) in edges:
        for s, d in [(src, dst), (dst, src)]:
            edge_rows.append({
                "graph_id":           gid,
                "src_id":             s,
                "dst_id":             d,
                "feat_connectivity_0": 0,
                "feat_connectivity_1": 1,
                "feat_connectivity_2": 0,
            })
    edges_df = pd.DataFrame(edge_rows)

    return nodes_df, edges_df


# Build all units
all_nodes, all_edges, graphs_rows = [], [], []
for uid, udata in UNIT_ROOMS.items():
    n_df, e_df = build_unit_dataframes(uid, udata)
    all_nodes.append(n_df)
    all_edges.append(e_df)
    graphs_rows.append({"graph_id": udata["graph_id"], "num_nodes": len(udata["rooms"])})

theedge_nodes_df  = pd.concat(all_nodes,  ignore_index=True)
theedge_edges_df  = pd.concat(all_edges,  ignore_index=True)
theedge_graphs_df = pd.DataFrame(graphs_rows)

print(f"TheEdge: {len(theedge_nodes_df)} nodes | {len(theedge_edges_df)} directed edges | {len(theedge_graphs_df)} graphs")
print()
print("Node label distribution:")
dist = theedge_nodes_df.groupby("label").size().reset_index(name="count")
dist["room_type"] = dist["label"].map(LABEL_NAMES)
print(dist.to_string(index=False))

In [ ]:
# MSD-compatible column order (no metadata)
MSD_NODE_COLS = [
    "graph_id", "node_id", "label",
    "feat_zoning_type_0", "feat_zoning_type_1", "feat_zoning_type_2", "feat_zoning_type_3",
    "feat_connectivity_0", "feat_connectivity_1", "feat_connectivity_2",
    "train_mask", "val_mask", "test_mask",
]

# Save metadata separately for post-prediction analysis
META_COLS = ["graph_id", "node_id", "_unit", "_room_name", "_floor", "_area"]
metadata_df = theedge_nodes_df[META_COLS].copy()
metadata_df.to_csv(EDGE_PATH / "rooms_metadata.csv", index=False)

# Export clean TheEdge CSVs
theedge_nodes_df[MSD_NODE_COLS].to_csv(EDGE_PATH / "nodes.csv", index=False)
theedge_edges_df.to_csv(EDGE_PATH / "edges.csv", index=False)
theedge_graphs_df.to_csv(EDGE_PATH / "graphs.csv", index=False)

print(f"Exported TheEdge dataset to: {EDGE_PATH}")
for f in ["graphs.csv", "nodes.csv", "edges.csv", "rooms_metadata.csv"]:
    p = EDGE_PATH / f
    print(f"  {f}: {p.stat().st_size:,} bytes")

## Section 4 — Combine with MSD Training Data

The Edge graphs (graph_ids 5001–5008) are appended to the MSD dataset.
MSD nodes keep their original masks. The Edge nodes have `test_mask=1`.  
GraphSAGE trains only on MSD `train_mask=1` nodes, then predicts on all `test_mask=1` nodes.

In [ ]:
msd_nodes  = pd.read_csv(MSD_PATH / "nodes.csv")
msd_edges  = pd.read_csv(MSD_PATH / "edges.csv")
msd_graphs = pd.read_csv(MSD_PATH / "graphs.csv")

print(f"MSD dataset:    {len(msd_nodes):>6} nodes | {len(msd_edges):>6} edges | {len(msd_graphs)} graphs")
print(f"TheEdge:        {len(theedge_nodes_df):>6} nodes | {len(theedge_edges_df):>6} edges | {len(theedge_graphs_df)} graphs")

# Verify column alignment
te_node_cols = set(theedge_nodes_df[MSD_NODE_COLS].columns)
msd_node_cols = set(msd_nodes.columns)
assert te_node_cols == msd_node_cols, f"Column mismatch: {te_node_cols ^ msd_node_cols}"

# Concatenate
comb_nodes  = pd.concat([msd_nodes,  theedge_nodes_df[MSD_NODE_COLS]], ignore_index=True)
comb_edges  = pd.concat([msd_edges,  theedge_edges_df],                ignore_index=True)
comb_graphs = pd.concat([msd_graphs, theedge_graphs_df],               ignore_index=True)

comb_nodes.to_csv(COMB_PATH / "nodes.csv", index=False)
comb_edges.to_csv(COMB_PATH / "edges.csv", index=False)
comb_graphs.to_csv(COMB_PATH / "graphs.csv", index=False)

print(f"\nCombined saved: {len(comb_nodes):>6} nodes | {len(comb_edges):>6} edges | {len(comb_graphs)} graphs")
print(f"  Train nodes: {int(comb_nodes['train_mask'].sum())}")
print(f"  Val nodes:   {int(comb_nodes['val_mask'].sum())}")
print(f"  Test nodes:  {int(comb_nodes['test_mask'].sum())} (includes {len(theedge_nodes_df)} TheEdge rooms)")

## Section 5 — GraphSAGE Node Classification

Replicates the S06-15 pipeline exactly, pointed at the combined MSD + TheEdge dataset.

In [ ]:
DATASET_PATH = COMB_PATH

PREDICTION_LEVEL  = "node"
TASK              = "classification"
GRAPH_LABEL_TYPE  = "categorical"
NODE_LABEL_TYPE   = "categorical"
EDGE_LABEL_TYPE   = "categorical"
HOLDOUT_GROUP_BY  = "label"

CONV         = "sage"
HIDDEN_DIMS  = (64, 64, 64)
ACTIVATION   = "relu"
DROPOUT      = 0.0
BATCH_NORM   = True
RESIDUAL     = False
POOLING      = "mean"

CROSS_VALIDATION = "holdout"
TRAIN_RATIO      = 0.7
VAL_RATIO        = 0.15
TEST_RATIO       = 0.15
RANDOM_STATE     = 42
SHUFFLE          = True

LEARNING_RATE          = 1e-2
BATCH_SIZE             = 10
EPOCHS                 = 50
OPTIMIZER              = "adamw"
WEIGHT_DECAY           = 0.01
GRADIENT_CLIP_NORM     = 1.0
EARLY_STOPPING         = True
EARLY_STOPPING_PATIENCE= 12
USE_GPU                = True

# Verify all three files exist
for f in ["graphs.csv", "nodes.csv", "edges.csv"]:
    p = DATASET_PATH / f
    print(f"  {f}: {'OK' if p.exists() else 'MISSING'}")

In [ ]:
pyg = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE,
)

print("Dataset loaded.")
print(json.dumps(pyg.Summary(), indent=2, default=str))

In [ ]:
pyg.SetHyperparameters(
    cv=CROSS_VALIDATION,
    split=(TRAIN_RATIO, VAL_RATIO, TEST_RATIO),
    random_state=RANDOM_STATE,
    shuffle=SHUFFLE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    optimizer=OPTIMIZER,
    gradient_clip_norm=GRADIENT_CLIP_NORM,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    use_gpu=USE_GPU,
    conv=CONV,
    hidden_dims=HIDDEN_DIMS,
    activation=ACTIVATION,
    dropout=DROPOUT,
    batch_norm=BATCH_NORM,
    residual=RESIDUAL,
    pooling=POOLING,
)
print("Hyperparameters set.")

In [ ]:
history = pyg.Train()
print("Training complete.")
print("Keys:", list(history.keys()))

In [ ]:
fig_hist = pyg.PlotHistory()
fig_hist.update_layout(width=900, height=500, title="Training History — Loss & Accuracy")
fig_hist.show(renderer=renderer)

In [ ]:
val_metrics  = pyg.Validate()
test_metrics = pyg.Test()

print("Validation metrics:")
print(pd.DataFrame({"metric": list(val_metrics.keys()), "value": list(val_metrics.values())}).to_string(index=False))
print()
print("Test metrics:")
print(pd.DataFrame({"metric": list(test_metrics.keys()), "value": list(test_metrics.values())}).to_string(index=False))

In [ ]:
fig_cm = pyg.PlotConfusionMatrix(split="test", normalize=False, title="Test Confusion Matrix (MSD + TheEdge test nodes)")
fig_cm.show(renderer=renderer)

In [ ]:
# Export predictions for all nodes (copy of S06-15 export function)
_ = pyg.Predict(split="all", return_probs=True, attach_to_data=True)

def _to_class_index(value):
    arr = np.asarray(value)
    arr = np.squeeze(arr)
    if arr.ndim == 0:  return int(arr)
    if arr.ndim == 1:
        if arr.size == 1:  return int(arr[0])
        return int(np.argmax(arr))
    raise ValueError(f"Cannot convert shape {arr.shape} to class index.")

def export_node_predictions(pyg_obj, output_csv):
    pred_report  = pyg_obj.Predict(split="all", return_probs=True, attach_to_data=True)
    pred_by_graph  = pred_report["pred"]
    y_true_by_graph= pred_report["y_true"]
    prob_by_graph  = pred_report.get("prob", None)

    rows = []
    for graph_idx, data in enumerate(pyg_obj.data_list):
        graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else graph_idx
        n = data.num_nodes
        graph_pred = np.asarray(pred_by_graph[graph_idx])
        graph_true = np.asarray(y_true_by_graph[graph_idx])
        graph_prob = np.asarray(prob_by_graph[graph_idx]) if prob_by_graph is not None else None

        train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, "train_mask") else None
        val_mask   = data.val_mask.detach().cpu().numpy()   if hasattr(data, "val_mask")   else None
        test_mask  = data.test_mask.detach().cpu().numpy()  if hasattr(data, "test_mask")  else None

        for node_idx in range(n):
            row = {
                "graph_id": graph_id,
                "node_id":  node_idx,
                "y_true":   _to_class_index(graph_true[node_idx]),
                "y_pred":   _to_class_index(graph_pred[node_idx]),
            }
            if graph_prob is not None:
                prob_i = np.asarray(graph_prob[node_idx]).squeeze()
                if prob_i.ndim == 1 and row["y_pred"] < prob_i.size:
                    row["confidence"] = float(prob_i[row["y_pred"]])
            if train_mask is not None: row["train_mask"] = bool(train_mask[node_idx])
            if val_mask   is not None: row["val_mask"]   = bool(val_mask[node_idx])
            if test_mask  is not None: row["test_mask"]  = bool(test_mask[node_idx])
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df


predictions_df = export_node_predictions(pyg, COMB_PATH / "node_predictions.csv")
print(f"Predictions exported: {len(predictions_df)} rows")
print(f"Saved to: {COMB_PATH / 'node_predictions.csv'}")

## Section 6 — Analysis: The Edge Predictions

Filter to The Edge nodes (graph_ids 5001–5008), merge room metadata, and analyse spatial patterns.

In [ ]:
THE_EDGE_GRAPH_IDS = [5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008]

te_pred = predictions_df[predictions_df["graph_id"].isin(THE_EDGE_GRAPH_IDS)].copy()

# Merge room metadata
meta = pd.read_csv(EDGE_PATH / "rooms_metadata.csv")
te_pred = te_pred.merge(meta, on=["graph_id", "node_id"], how="left")

# Add label names and correctness flag
te_pred["true_name"] = te_pred["y_true"].map(LABEL_NAMES)
te_pred["pred_name"] = te_pred["y_pred"].map(LABEL_NAMES)
te_pred["correct"]   = te_pred["y_true"] == te_pred["y_pred"]

accuracy = te_pred["correct"].mean()
print(f"The Edge — overall accuracy: {accuracy:.1%}  ({te_pred['correct'].sum()}/{len(te_pred)} rooms correct)")
print()
print("Summary by unit:")
unit_acc = te_pred.groupby("_unit")["correct"].agg(["sum", "count", "mean"]).reset_index()
unit_acc.columns = ["Unit", "Correct", "Total", "Accuracy"]
unit_acc["Accuracy"] = unit_acc["Accuracy"].map("{:.1%}".format)
print(unit_acc.to_string(index=False))

In [ ]:
# Full prediction table — sorted by unit then node_id
display_cols = ["_unit", "_room_name", "_floor", "y_true", "true_name", "y_pred", "pred_name", "correct"]
report = te_pred[display_cols].sort_values(["_unit", "node_id"]).copy()
report.columns = ["Unit", "Room", "Floor", "True Label", "True Type", "Pred Label", "Pred Type", "Correct"]

def color_correct(val):
    color = "#c8f7c5" if val else "#ffc8c8"
    return f"background-color: {color}"

report.style.applymap(color_correct, subset=["Correct"])

In [ ]:
# Per-unit accuracy bar chart
unit_stats = te_pred.groupby("_unit").agg(
    correct=("correct", "sum"),
    total=("correct", "count")
).reset_index()
unit_stats["accuracy"] = unit_stats["correct"] / unit_stats["total"]
unit_stats["unit_type"] = unit_stats["_unit"].map({u: UNIT_ROOMS[u]["unit_type"] for u in UNIT_ROOMS})

fig = px.bar(
    unit_stats,
    x="_unit", y="accuracy",
    text=unit_stats["accuracy"].map("{:.0%}".format),
    color="accuracy", color_continuous_scale="RdYlGn",
    range_color=[0, 1],
    labels={"_unit": "Unit", "accuracy": "Prediction Accuracy"},
    title="GraphSAGE Node Classification Accuracy per Unit — The Edge",
)
fig.update_traces(textposition="outside")
fig.update_layout(width=800, height=450, coloraxis_showscale=False)
fig.show(renderer=renderer)

In [ ]:
# Accuracy by room label type
label_stats = te_pred.groupby(["y_true", "true_name"]).agg(
    correct=("correct", "sum"),
    total=("correct", "count")
).reset_index()
label_stats["accuracy"] = label_stats["correct"] / label_stats["total"]

fig2 = px.bar(
    label_stats.sort_values("y_true"),
    x="true_name", y="accuracy",
    text=label_stats.sort_values("y_true")["accuracy"].map("{:.0%}".format),
    color="accuracy", color_continuous_scale="RdYlGn",
    range_color=[0, 1],
    labels={"true_name": "Room Type (Ground Truth)", "accuracy": "Accuracy"},
    title="Prediction Accuracy by Room Type — The Edge",
)
fig2.update_xaxes(tickangle=30)
fig2.update_traces(textposition="outside")
fig2.update_layout(width=1000, height=500, coloraxis_showscale=False)
fig2.show(renderer=renderer)

In [ ]:
# Confusion among TheEdge test nodes
wrong = te_pred[~te_pred["correct"]].copy()
if len(wrong) > 0:
    print(f"\nMis-predicted rooms ({len(wrong)} total):\n")
    w_display = wrong[["_unit", "_room_name", "_floor", "true_name", "pred_name"]].copy()
    w_display.columns = ["Unit", "Room", "Floor", "Should be", "Predicted as"]
    print(w_display.sort_values(["Unit"]).to_string(index=False))
else:
    print("All rooms correctly predicted!")

## Section 7 — Spatial Interpretation

### What to look for

**Expected correct predictions (model strength):**
- Cocina / Kitchenette → label 5 (Kitchen): these have a unique structural signature — they connect to both day zones and service zones simultaneously
- Baño / Baño Principal → label 7 (Bathroom): bathrooms are graph "leaf" nodes attached only to their bedroom, which is a strong structural signal
- Estar / Comedor → label 0 (Day zone): high-degree nodes adjacent to kitchen and terrace are consistently classified as living spaces in MSD

**Expected mis-predictions (interesting for analysis):**
- **Vestidor → label 3 (Walk-in Closet)**: Only 42 of 15,914 MSD training nodes carry label 3. The model has almost no examples to learn from. Expect vestidores to be predicted as label 1 (bedroom) or label 2 (master bedroom), because they share the same neighbourhood topology: attached to a single bedroom, with no service-zone neighbors.
- **Terraza / Solárium / Jardín → label 0 (Day zone)**: Outdoor spaces don't exist in Swiss apartments. The model predicts them as day zone rooms, which is *architecturally reasonable*, but is confusing the model because terrace nodes have fewer connections than Swiss living rooms (dead-end topology).
- **Toilette → label 6 (WC)**: Small rooms adjacent only to the kitchen are rare in MSD (WCs normally attach to corridors). Expect some confusion with label 7 (bathroom) or label 5 (kitchen/utility).

### What the results reveal about The Edge vs. Swiss typology

The Edge is a **luxury residential building** with spatial features that differ from Swiss standard apartments:
1. **Vestidores (walk-in closets)** — a luxury feature absent from MSD training data
2. **Private terraces and gardens** — much larger than MSD balconies; classified as day zones but with atypical topology
3. **Service bedrooms** — a live-in service program not present in Swiss apartments
4. **Rooftop soláriums** — open outdoor amenity spaces with no MSD equivalent

**Correctly predicted rooms** validate that the core residential graph structure (living → dining → kitchen, bedroom → bathroom) is universal across Swiss and Uruguayan luxury typologies.  
**Incorrectly predicted rooms** pinpoint the architectural features that distinguish The Edge from the dataset's cultural and programmatic context.